# Published Benchmarks

Two games from the literature with partial information and an exact solution of the discretised game. In the
broker-trader game the broker's information is contained in the trader's (nested information); in the major-minor
game neither player's information contains the other's. The learned method is run at the small budget of
`two_traders.ipynb` (5,000 paths), so that the notebook runs in about 10 minutes on a CPU.

In [ ]:
import sys, numpy as np, torch, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
from partial_info_games import Solver, first_order_residual, relative_l2, use_device, validation_paths
from partial_info_games import use_style, plot_controls
use_style()
DEVICE = 'cpu'                     # 'mps' (Apple GPU) or 'cuda' to use a GPU
device = use_device(DEVICE)
to_players = lambda controls: [torch.tensor(controls[:, i], dtype=torch.float32) for i in range(2)]
print('device:', device)

from partial_info_games.games import BrokerTrader, MajorMinor

## 1. Broker and informed trader (Wu and Jaimungal, 2024)

An informed trader observes a mean-reverting signal $\alpha$ and the price, and trades $\eta$ with a broker, who observes
the price only and trades $\nu$ on the exchange; here without transient impact. The price, the signal and the
inventories follow

$$
\begin{aligned}
dz_t &= \alpha_t\,dt + \sigma\,dW_t, \\[4pt]
d\alpha_t &= -\kappa\,\alpha_t\,dt + \sigma_\alpha\,dB_t, \\[4pt]
dQ^I_t &= \eta_t\,dt, \qquad dQ^B_t = (\nu_t - \eta_t)\,dt .
\end{aligned}
$$

The trader and the broker maximize

$$
\begin{aligned}
J^I &= \mathbb{E}\Big[ \int_0^T \big( Q^I_t\,\alpha_t - b\,\eta_t^2 - r_I\,(Q^I_t)^2 \big)\,dt - \psi\,(Q^I_T)^2 \Big], \\[4pt]
J^B &= \mathbb{E}\Big[ \int_0^T \big( Q^B_t\,\alpha_t - a\,\nu_t^2 - r_B\,(Q^B_t)^2 \big)\,dt - \phi\,(Q^B_T)^2 \Big].
\end{aligned}
$$

The exact solution is the trader's discrete LQR and the broker's Kalman filter of $(\alpha, Q^I)$ followed by a discrete LQR.

In [ ]:
game = BrokerTrader()
solver = Solver(game, damping=0.7, steps=300, batch=1000, n_paths=5000)
exact = to_players(game.exact(solver.validation))
learned, _ = solver.run(10, average_from=4, reference=exact)
print(f'relative L2 error: trader {relative_l2(learned[0], exact[0]):.3f}, broker {relative_l2(learned[1], exact[1]):.3f}')
plot_controls(learned, exact, game.dt, ['informed trader', 'broker'])

## 2. Major and minor player (Firoozi and Caines, 2020)

A major player and a minor player each observe their own state through noise; the minor player also observes the
major player's state through noise, and its dynamics depend on that state. The states and the observations follow

$$
\begin{aligned}
dx^0_t &= (A_0 x^0_t + B_0 u^0_t)\,dt + D_0\,dw^0_t, & dy^0_t &= L_0 x^0_t\,dt + s_0\,dv^0_t, \\[4pt]
dx^1_t &= (A_1 x^1_t + G x^0_t + B_1 u^1_t)\,dt + D_1\,dw^1_t, & dy^1_t &= (L_1 x^1_t,\, L_2 x^0_t)\,dt + s_1\,dv^1_t .
\end{aligned}
$$

The major and the minor player minimize the discounted costs

$$
\begin{aligned}
&\mathbb{E} \int_0^T e^{-\rho t} \big[ Q_0\,(x^0_t - \eta_0)^2 + R_0\,(u^0_t)^2 \big]\,dt, \\[4pt]
&\mathbb{E} \int_0^T e^{-\rho t} \big[ Q_1\,(x^1_t - H x^0_t - \eta_1)^2 + R_1\,(u^1_t)^2 \big]\,dt .
\end{aligned}
$$

The minor player must estimate the major player's own estimate of $x^0$. The exact solution is the discrete Kalman
filter and discounted LQR of each player.

In [ ]:
game = MajorMinor()
solver = Solver(game, damping=0.7, steps=300, batch=1000, n_paths=5000)
exact = to_players(game.exact(solver.validation))
learned, _ = solver.run(6, average_from=3, reference=exact)
print(f'relative L2 error: major {relative_l2(learned[0], exact[0]):.3f}, minor {relative_l2(learned[1], exact[1]):.3f}')
plot_controls(learned, exact, game.dt, ['major player', 'minor player'])